In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

keras.utils.set_random_seed(42)
print(tf.config.list_physical_devices("GPU"))

In [ ]:
!wget -q -O driving_data.zip https://d17h27t6h515a5.cloudfront.net/topher/2016/December/584f6edd_data/data.zip
!unzip -q -o driving_data.zip
data_folder = "data"
print(os.listdir(data_folder))

In [ ]:
driving_log = pd.read_csv(os.path.join(data_folder, "driving_log.csv"))
driving_log.columns = [column.strip() for column in driving_log.columns]
for camera in ["center", "left", "right"]:
    driving_log[camera] = driving_log[camera].str.strip()

print(driving_log.shape)
display(driving_log.head())

plt.figure(figsize=(9, 4))
plt.hist(driving_log["steering"], bins=50, color="steelblue")
plt.title("Steering angles in the raw log")
plt.xlabel("steering angle")
plt.show()

In [ ]:
straight_rows = driving_log[driving_log["steering"].abs() < 0.02]
turning_rows = driving_log[driving_log["steering"].abs() >= 0.02]
balanced_log = pd.concat([turning_rows, straight_rows.sample(frac=0.3, random_state=42)]).sample(frac=1, random_state=42).reset_index(drop=True)

print("straight frames:", len(straight_rows), "turning frames:", len(turning_rows), "kept:", len(balanced_log))
plt.figure(figsize=(9, 4))
plt.hist(balanced_log["steering"], bins=50, color="darkorange")
plt.title("Steering angles after balancing")
plt.xlabel("steering angle")
plt.show()

In [ ]:
def load_rgb_image(relative_path):
    image = cv2.imread(os.path.join(data_folder, relative_path))
    if image is None:
        return None
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

def preprocess_image(image):
    cropped = image[60:135, :, :]
    resized = cv2.resize(cropped, (200, 66), interpolation=cv2.INTER_AREA)
    blurred = cv2.GaussianBlur(resized, (3, 3), 0)
    return cv2.cvtColor(blurred, cv2.COLOR_RGB2YUV)

sample_row = balanced_log.iloc[0]
fig, axes = plt.subplots(1, 3, figsize=(16, 3))
for axis, camera in zip(axes, ["left", "center", "right"]):
    axis.imshow(load_rgb_image(sample_row[camera]))
    axis.set_title(f"{camera} camera")
    axis.axis("off")
plt.show()

original = load_rgb_image(sample_row["center"])
cropped = original[60:135, :, :]
resized = cv2.resize(cropped, (200, 66), interpolation=cv2.INTER_AREA)
yuv_image = preprocess_image(original)
fig, axes = plt.subplots(1, 4, figsize=(18, 3))
for axis, image, title in zip(axes, [original, cropped, resized, yuv_image], ["original 160x320", "cropped sky and hood", "resized 66x200", "blurred + YUV"]):
    axis.imshow(image)
    axis.set_title(title)
    axis.axis("off")
plt.show()

In [ ]:
steering_correction = 0.2
camera_offsets = [("center", 0.0), ("left", steering_correction), ("right", -steering_correction)]
image_list, angle_list = [], []

for _, row in balanced_log.iterrows():
    for camera, offset in camera_offsets:
        image = load_rgb_image(row[camera])
        if image is None:
            continue
        image_list.append(preprocess_image(image))
        angle_list.append(row["steering"] + offset)

road_images = np.array(image_list, dtype=np.uint8)
steering_angles = np.array(angle_list, dtype=np.float32)
del image_list, angle_list
print(road_images.shape, steering_angles.shape)

In [ ]:
train_images, val_images, train_angles, val_angles = train_test_split(road_images, steering_angles, test_size=0.2, random_state=42)

def augment_frame(image, angle):
    image = tf.cast(image, tf.float32)
    flip = tf.random.uniform(()) > 0.5
    image = tf.cond(flip, lambda: tf.image.flip_left_right(image), lambda: image)
    angle = tf.cond(flip, lambda: -angle, lambda: angle)
    brightness = tf.random.uniform((), 0.7, 1.3)
    image = tf.concat([tf.clip_by_value(image[..., :1] * brightness, 0.0, 255.0), image[..., 1:]], axis=-1)
    return image, angle

def cast_frame(image, angle):
    return tf.cast(image, tf.float32), angle

train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_angles)).shuffle(5000).map(augment_frame, num_parallel_calls=tf.data.AUTOTUNE).batch(64).prefetch(tf.data.AUTOTUNE)
val_dataset = tf.data.Dataset.from_tensor_slices((val_images, val_angles)).map(cast_frame).batch(64).prefetch(tf.data.AUTOTUNE)
print(len(train_images), len(val_images))

In [ ]:
steering_model = keras.Sequential([
    layers.Input(shape=(66, 200, 3)),
    layers.Rescaling(1.0 / 127.5, offset=-1.0),
    layers.Conv2D(24, 5, strides=2, activation="elu"),
    layers.Conv2D(36, 5, strides=2, activation="elu"),
    layers.Conv2D(48, 5, strides=2, activation="elu"),
    layers.Conv2D(64, 3, activation="elu"),
    layers.Conv2D(64, 3, activation="elu"),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(100, activation="elu"),
    layers.Dense(50, activation="elu"),
    layers.Dense(10, activation="elu"),
    layers.Dense(1)
], name="nvidia_pilotnet")

steering_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="mse", metrics=["mae"])
steering_model.summary()

In [ ]:
steering_history = steering_model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)
    ]
)

In [ ]:
predicted_angles = steering_model.predict(val_dataset, verbose=0).flatten()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].plot(steering_history.history["loss"], label="train")
axes[0].plot(steering_history.history["val_loss"], label="validation")
axes[0].set_title("Steering MSE")
axes[0].set_xlabel("epoch")
axes[0].legend()
axes[1].scatter(val_angles, predicted_angles, s=4, alpha=0.4)
axes[1].plot([-1, 1], [-1, 1], color="red")
axes[1].set_xlabel("true steering")
axes[1].set_ylabel("predicted steering")
axes[1].set_title("Predicted vs true on validation")
plt.show()

fig, axes = plt.subplots(3, 3, figsize=(15, 7))
for axis, index in zip(axes.flatten(), np.random.choice(len(val_images), 9, replace=False)):
    axis.imshow(cv2.cvtColor(val_images[index], cv2.COLOR_YUV2RGB))
    axis.set_title(f"true {val_angles[index]:.2f}  predicted {predicted_angles[index]:.2f}")
    axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
steering_model.save("nvidia_steering_model.keras")
del road_images, train_images

In [ ]:
!wget -q -O har_dataset.zip "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
with zipfile.ZipFile("har_dataset.zip") as archive:
    archive.extractall("har_raw")
inner_zip = os.path.join("har_raw", "UCI HAR Dataset.zip")
if os.path.exists(inner_zip):
    with zipfile.ZipFile(inner_zip) as archive:
        archive.extractall("har_raw")
har_folder = os.path.join("har_raw", "UCI HAR Dataset")
print(os.listdir(har_folder))

In [ ]:
signal_names = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z"
]

def load_har_split(split):
    signals = [np.loadtxt(os.path.join(har_folder, split, "Inertial Signals", f"{name}_{split}.txt")) for name in signal_names]
    windows = np.stack(signals, axis=-1).astype(np.float32)
    labels = np.loadtxt(os.path.join(har_folder, split, f"y_{split}.txt")).astype(int) - 1
    return windows, labels

har_train_windows, har_train_labels = load_har_split("train")
har_test_windows, har_test_labels = load_har_split("test")
with open(os.path.join(har_folder, "activity_labels.txt")) as label_file:
    activity_names = [line.split()[1] for line in label_file if line.strip()]

print(har_train_windows.shape, har_test_windows.shape)
print(activity_names)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 7))
for axis, activity_id in zip(axes.flatten(), range(len(activity_names))):
    example_window = har_train_windows[np.where(har_train_labels == activity_id)[0][0]]
    for channel, name in enumerate(signal_names[6:], start=6):
        axis.plot(example_window[:, channel], label=name)
    axis.set_title(activity_names[activity_id])
axes[0, 0].legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(9, 4))
sns.countplot(x=[activity_names[label] for label in har_train_labels])
plt.xticks(rotation=30)
plt.title("Training windows per activity")
plt.show()

In [ ]:
channel_mean = har_train_windows.mean(axis=(0, 1))
channel_std = har_train_windows.std(axis=(0, 1))
har_train_scaled = (har_train_windows - channel_mean) / channel_std
har_test_scaled = (har_test_windows - channel_mean) / channel_std

har_model = keras.Sequential([
    layers.Input(shape=(128, 9)),
    layers.Conv1D(64, 5, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv1D(64, 5, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling1D(2),
    layers.Dropout(0.3),
    layers.Conv1D(128, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.Conv1D(128, 3, padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.GlobalAveragePooling1D(),
    layers.Dropout(0.4),
    layers.Dense(100, activation="relu"),
    layers.Dense(len(activity_names), activation="softmax")
], name="har_cnn")

har_model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
har_model.summary()

In [ ]:
har_history = har_model.fit(
    har_train_scaled, har_train_labels,
    validation_split=0.2,
    epochs=40,
    batch_size=64,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)
    ]
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(har_history.history["loss"], label="train")
axes[0].plot(har_history.history["val_loss"], label="validation")
axes[0].set_title("HAR loss")
axes[0].legend()
axes[1].plot(har_history.history["accuracy"], label="train")
axes[1].plot(har_history.history["val_accuracy"], label="validation")
axes[1].set_title("HAR accuracy")
axes[1].legend()
plt.show()

In [ ]:
test_loss, test_accuracy = har_model.evaluate(har_test_scaled, har_test_labels, verbose=0)
print(f"test accuracy: {test_accuracy:.4f}")

har_predictions = har_model.predict(har_test_scaled, verbose=0).argmax(axis=1)
print(classification_report(har_test_labels, har_predictions, target_names=activity_names, digits=4))

plt.figure(figsize=(9, 7))
sns.heatmap(confusion_matrix(har_test_labels, har_predictions), annot=True, fmt="d", cmap="Blues", xticklabels=activity_names, yticklabels=activity_names)
plt.xlabel("predicted")
plt.ylabel("actual")
plt.title("HAR confusion matrix")
plt.show()

har_model.save("har_cnn_model.keras")